# Travel agency's reviews - classification with BERT

Implement and evaluate a classifier of user reviews with BERT.

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
import pandas as pd

reviews = pd.read_csv('https://raw.githubusercontent.com/mlcollege/natural-language-processing/master/data/en_reviews.csv', sep='\t', header=None, names =['rating', 'text'])
reviews[35:45]

,rating,text
35,5,I bought the cheapest tickets through this ser...
36,5,Such a pleasure to know that you will be prope...
37,5,I always use this website to look for flights ...
38,2,A startup that finds discount flight tickets '...
39,5,"Excellent customer service, fast and kind. Wan..."
40,4,very good service from Quan Costa to help me w...
41,3,.@Skypickercom Finds Cheap Flights 'Hidden' On...
42,5,I have a problem with my tickets skypicker don...
43,4,Even though it took a bit time untill an agent...
44,5,Today I had a great experience with one of Kiw...


## Preparation of train and test data sets
Separate and rename target values.

In [ ]:
# Extract the 'rating' column as the target variable for classification
target = reviews['rating']
# Extract the 'text' column as the input data for the model
data = reviews['text']

# Print the first 5 entries of the data and target to verify the extraction
print(data[:5])
print(target[:5])

0    A voucher to nowhere #skypickerfail 2400 out o...
1    I booked with Kiwi for the first time, just a ...
2    I would like to say THANKS YOU for your custom...
3    I just noticed 2 hours before my flight that I...
4    This is the first time I have dealt with Skypi...
Name: text, dtype: object
0    2
1    5
2    5
3    5
4    2
Name: rating, dtype: int64


Import the BERT model and tokenizer

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
# Import BertTokenizer for converting text into a format suitable for BERT.
# Import BertForSequenceClassification, a BERT model fine-tuned for classification tasks.

In [ ]:
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=5)
bert_model.to(device)

Split the data to train and test parts.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(data, target, test_size=0.1)
print('Train size: {}'.format(len(X_train)))
print('Test size: {}'.format(len(X_test)))

Tokenize the documents and create attention masks.

In [ ]:
train_encodings = bert_tokenizer(
    list(X_train),
    padding='max_length',
    max_length=64,
    truncation=True,
    return_tensors='pt'
) # Tokenize training texts, padding them to max_length=64, truncating longer ones, and returning PyTorch tensors.

test_encodings = bert_tokenizer(
    list(X_test),
    padding='max_length',
    max_length=64,
    truncation=True,
    return_tensors='pt'
) # Tokenize test texts, similar to training texts.

# Print the shapes of the input_ids tensors to verify the dimensions after tokenization.
print(train_encodings['input_ids'].shape)
print(test_encodings['input_ids'].shape)

Prepare target labels.

In [ ]:
y_train = torch.tensor((y_train - 1).values, dtype=torch.long)
y_test = torch.tensor((y_test - 1).values, dtype=torch.long)
# The target labels (ratings) are originally 1-5.
# For BERT's `num_labels` parameter (which was set to 5), the labels should be 0-indexed (0-4).
# Subtracting 1 from each rating converts them to the 0-indexed format.
# `dtype=torch.long` is specified because PyTorch's cross-entropy loss function expects integer labels of type Long.

Prepare data loaders and optimizer.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
from torch.optim import AdamW

# Create TensorDatasets from the tokenized inputs and target labels
# A TensorDataset wraps tensors, providing a way to access corresponding slices
# along the first dimension of each tensor.
train_dataset = TensorDataset(train_encodings['input_ids'], train_encodings['attention_mask'], y_train)
test_dataset = TensorDataset(test_encodings['input_ids'], test_encodings['attention_mask'], y_test)

# Create DataLoaders for efficient batching and iteration over the datasets
# train_loader shuffles the data for better generalization during training
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
# test_loader does not shuffle as order is not important for evaluation
test_loader = DataLoader(test_dataset, batch_size=32)

# Initialize the AdamW optimizer for updating model parameters
# AdamW is a variant of Adam that decouples weight decay from the gradient update
# lr (learning rate) is set to 2e-5, a common value for fine-tuning BERT
# eps (epsilon) is a small value for numerical stability in the optimizer
optimizer = AdamW(bert_model.parameters(), lr=2e-5, eps=1e-08)

In [ ]:
from tqdm.notebook import tqdm

# Loop through the specified number of epochs (here, 3 epochs)
for epoch in range(3):
    # Set the model to training mode
    bert_model.train()
    total_loss = 0
    correct = 0
    total = 0

    # Iterate over batches from the training data loader
    # tqdm.notebook.tqdm provides a progress bar for the training loop
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/3 - Training"):
        # Move input IDs, attention masks, and labels to the specified device (GPU/CPU)
        input_ids, attention_mask, labels = [b.to(device) for b in batch]

        # Zero out the gradients from the previous iteration
        optimizer.zero_grad()

        # Perform a forward pass: get model outputs (logits and loss)
        outputs = bert_model(input_ids, attention_mask=attention_mask, labels=labels)

        # Backpropagate the loss to compute gradients
        outputs.loss.backward()

        # Update model parameters using the optimizer
        optimizer.step()

        # Accumulate total loss and calculate training accuracy
        total_loss += outputs.loss.item()
        correct += (outputs.logits.argmax(dim=1) == labels).sum().item()
        total += len(labels)

    # --- Validation Phase ---
    # Set the model to evaluation mode
    # This disables dropout and freezes batch normalization layers for consistent evaluation
    bert_model.eval()
    val_correct = 0
    val_total = 0
    # Disable gradient calculations for validation to save memory and speed up computation
    with torch.no_grad():
        # Iterate over batches from the test data loader
        for batch in test_loader:
            # Move input IDs, attention masks, and labels to the device
            input_ids, attention_mask, labels = [b.to(device) for b in batch]

            # Perform a forward pass to get model outputs (logits)
            # Labels are passed to allow internal loss calculation, though accuracy is used here
            outputs = bert_model(input_ids, attention_mask=attention_mask, labels=labels)

            # Calculate validation accuracy
            val_correct += (outputs.logits.argmax(dim=1) == labels).sum().item()
            val_total += len(labels)

    # Print training loss, training accuracy, and validation accuracy for the current epoch
    print(f"Epoch {epoch+1}/3 - loss: {total_loss/len(train_loader):.4f} - accuracy: {correct/total:.4f} - val_accuracy: {val_correct/val_total:.4f}")

## Evaluate the model

In [ ]:
import numpy as np
from sklearn import metrics
from sklearn.metrics import accuracy_score

# Set the model to evaluation mode
# This is crucial for consistent behavior during inference (e.g., disables dropout)
bert_model.eval()

all_preds = []
# Disable gradient calculation for inference to save memory and speed up computation
with torch.no_grad():
    # Iterate through the test data loader to get predictions for all test samples
    for batch in test_loader:
        # Move input IDs and attention masks to the specified device (GPU/CPU)
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        # Perform a forward pass to get the model's output logits
        outputs = bert_model(input_ids, attention_mask=attention_mask)
        # Get the predicted class by finding the index with the maximum logit value (argmax)
        # Move predictions to CPU and convert to a NumPy array
        all_preds.extend(outputs.logits.argmax(dim=1).cpu().numpy())

# Convert the list of all predictions to a NumPy array
y_pred_class = np.array(all_preds)
# Convert the true test labels (y_test) to a NumPy array for scikit-learn metrics
y_test_class = y_test.numpy()

# Calculate and print the overall test accuracy
print("Test accuracy: {:.4f}".format(accuracy_score(y_test_class, y_pred_class)))
print()
# Generate and print a detailed classification report
# This includes precision, recall, f1-score, and support for each class, as well as overall averages.
print(metrics.classification_report(y_test_class, y_pred_class, digits=4))